# Getting Started 2: Training a Lipschitz Network


Here we train a small 1-Lipschitz model on a synthetic classification problem. We rely on `TauCategoricalCrossentropy`, which is tailored to Lipschitz networks.


In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from deel.lip.layers import SpectralDense, GroupSort2
from deel.lip.losses import TauCategoricalCrossentropy
from deel.lip.model import Sequential


In [ ]:
torch.manual_seed(1)
num_samples = 512
num_features = 32
num_classes = 3

inputs = torch.randn(num_samples, num_features)
weights = torch.randn(num_features, num_classes)
labels = torch.softmax(inputs @ weights, dim=-1)
labels = torch.argmax(labels, dim=-1)

dataset = TensorDataset(inputs, labels)
loader = DataLoader(dataset, batch_size=64, shuffle=True)


In [ ]:
model = Sequential(
    SpectralDense(num_features, 64, activation="relu", use_bias=False),
    GroupSort2(),
    SpectralDense(64, num_classes, use_bias=False),
)
loss_fn = TauCategoricalCrossentropy(tau=2.0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
def one_hot(targets, num_classes):
    return torch.nn.functional.one_hot(targets, num_classes=num_classes).float()

for epoch in range(5):
    total_loss = 0.0
    for batch_inputs, batch_labels in loader:
        optimizer.zero_grad()
        logits = model(batch_inputs)
        loss = loss_fn(one_hot(batch_labels, num_classes), logits)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch_inputs.size(0)
    print(f"Epoch {epoch + 1}: loss={(total_loss / len(dataset)):.4f}")


In [ ]:
with torch.no_grad():
    logits = model(inputs)
    preds = torch.argmax(logits, dim=-1)
    accuracy = (preds == labels).float().mean()
print(f"Training accuracy: {accuracy:.3f}")
